# Carga del catálogo SQLite: 250 CVE por año

Este notebook consulta NVD y carga los catálogos completos de CWE, CAPEC y ExploitDB. La muestra contiene **250 CVE por año de publicación (2023–2026)**, seleccionados sin reemplazo de la población elegible completa. La ejecución sustituye el contenido previo dentro de una sola transacción y guarda un respaldo.

Los campos y relaciones del esquema v1 se conservan. `NVDExample.json` es una muestra de prueba y no participa en esta selección. La clave de `.env` se utiliza internamente, sin mostrar su valor.

## 1. Configuración y rutas

Ejecutar con el Python de `.venv` después de instalar `preprosesamiento/requirements.txt`. `SEMILLA=42` hace reproducible el muestreo. `REFRESCAR_NVD=False` reutiliza o reanuda la descarga y su fecha de corte; cambiarlo a `True` crea una descarga nueva. Una semilla distinta cambia la muestra sin volver a consultar las páginas disponibles.

In [1]:
import json
import sqlite3
import sys
from pathlib import Path
from contextlib import closing
from IPython.display import display, HTML

PROYECTO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "preprosesamiento/sqlite/schema.sql").is_file()), None)
if PROYECTO is None:
    raise RuntimeError("Abrir el notebook desde el repositorio o una de sus subcarpetas")
SQLITE = PROYECTO / "preprosesamiento/sqlite"
FUENTES = SQLITE / "fuentes"
SALIDAS = SQLITE / "salidas"
BASE = SQLITE / "catalogo.sqlite3"
sys.path.insert(0, str(SQLITE))
from carga_catalogo import (YEARS, INSERT_COLUMNS, download_and_select,
                            prepare_catalogue, load_catalogue)
from revisar_fuentes import review

ANIOS = YEARS
CUOTA_ANUAL = 250
SEMILLA = 42
REFRESCAR_NVD = False
print({"anios": ANIOS, "cuota_anual": CUOTA_ANUAL, "semilla": SEMILLA,
       "refrescar_nvd": REFRESCAR_NVD, "base": BASE.name})

{'anios': (2023, 2024, 2025, 2026), 'cuota_anual': 250, 'semilla': 42, 'refrescar_nvd': False, 'base': 'catalogo.sqlite3'}


## 2. Lectura y diagnóstico de las fuentes locales

El lector respeta comillas CSV. Los archivos MITRE tienen una columna extra vacía y CAPEC usa el encabezado `'ID`. La auditoría muestra duplicados y cobertura antes de preparar INSERTS. Los originales se mantienen sin modificaciones.

In [2]:
revision = review()
print("Archivos y hashes:")
for nombre, datos in revision["files"].items():
    print(nombre, "registros:", datos["rows"], "sha256:", datos["sha256"])
print("CWE únicos:", revision["cwe"]["unique_ids"])
print("CAPEC:", revision["capec"]["rows"], "pares únicos:", revision["capec"]["unique_relations"])
print("ExploitDB únicos:", revision["exploit_db"]["unique_ids"])
print("NVDExample: fixture de cinco registros; no se inserta en la muestra anual.")

Archivos y hashes:
capec.csv registros: 559 sha256: 6ba3971aefee4c2ce4d29c2b50b85acab15016133f1f60a765229d780af4f44f
cwe_hardware.csv registros: 110 sha256: 54fed63711677c2e7aed1164844da5ca2377cc7fac4d5e09e6ffad8c77d17b27
cwe_software.csv registros: 399 sha256: f4a30c7eaf1c8a4a6045b3be48571f74ee6e23cdbd768e3c40f48f5e67e02101
files_exploits.csv registros: 47170 sha256: 49ed79cb1bf0878931793318485d21cdc947f1d91ef8accbd09d5d4bf99991b8
NVDExample.json registros: 5 sha256: 44802bbaabe6106d96550fe1c5a8ca52869b815f165822ecb235cf88b3febf92
CWE únicos: 501
CAPEC: 559 pares únicos: 1212
ExploitDB únicos: 46698
NVDExample: fixture de cinco registros; no se inserta en la muestra anual.


## 3. Consulta completa de NVD, filtros y caché

Se recorren ventanas mensuales y todas sus páginas, con seis segundos entre solicitudes y reintentos transitorios. La fecha de corte de 2026 se fija al comenzar la descarga. Cada página se guarda comprimida y con SHA-256 para poder reanudar una interrupción.

Se admiten estados `Analyzed`/`Modified`, una descripción no vacía, alguna métrica **3.0 o 3.1 NETWORK** y una referencia HTTP(S) con `Vendor Advisory`, o `Exploit` hacia GitHub/GitLab/Bitbucket/ExploitDB. Una métrica LOCAL en otra versión no excluye un CVE que tenga NETWORK. Se conservan todas sus métricas y referencias.

In [3]:
seleccion = download_and_select(
    PROYECTO, SALIDAS, years=ANIOS, quota=CUOTA_ANUAL,
    seed=SEMILLA, refresh=REFRESCAR_NVD,
)
print("Fecha de corte UTC:", seleccion.report["cutoff"])
print("Páginas en caché:", seleccion.report["cached_pages"])

{"phase": "download", "year": 2023, "month": 1, "index": 0, "records": 2000, "cached_pages": 1}


{"phase": "download", "year": 2023, "month": 1, "index": 2000, "records": 565, "cached_pages": 2}


{"phase": "wait", "seconds": 4.614359245999367}


{"phase": "download", "year": 2023, "month": 2, "index": 0, "records": 2000, "cached_pages": 3}


{"phase": "download", "year": 2023, "month": 2, "index": 2000, "records": 474, "cached_pages": 4}


...

{"phase": "download", "year": 2026, "month": 9, "index": 14000, "records": 1074, "cached_pages": 123}


{"phase": "wait", "seconds": 2.982469022999794}


{"phase": "download", "year": 2026, "month": 10, "index": 0, "records": 1417, "cached_pages": 124}


{"phase": "selection", "year": 2026, "eligible": 21920, "selected": 250}


Fecha de corte UTC: 2026-10-06T03:43:16.289Z
Páginas en caché: 124


## 4. Selección aleatoria independiente por año

Los IDs se deduplican antes de filtrar, conservando su observación más reciente. Se ordena la población elegible por ID y se utiliza `random.Random(SEMILLA + año).sample(..., 250)`. El año corresponde a `published`, aunque difiera del año del identificador. Si una cuota es insuficiente, el proceso se detiene antes de modificar SQLite.

In [4]:
filas = []
for anio in ANIOS:
    detalle = seleccion.report["annual"][str(anio)]
    filas.append(f"<tr><td>{anio}</td><td>{detalle['records_downloaded']}</td>"
                 f"<td>{detalle['eligible']}</td><td>{detalle['selected']}</td></tr>")
    print(anio, "descartes:", detalle["discarded"],
          "duplicados:", detalle["duplicate_observations"])
display(HTML("<table><tr><th>Año</th><th>Consultados</th><th>Elegibles</th>"
             "<th>Seleccionados</th></tr>" + "".join(filas) + "</table>"))
assert len(seleccion.records) == len(ANIOS) * CUOTA_ANUAL
assert len({registro["id"] for registro in seleccion.records}) == len(seleccion.records)

2023 descartes: {'estado': 2133, 'referencia': 10145, 'cvss_network': 6982} duplicados: 0
2024 descartes: {'referencia': 10702, 'cvss_network': 10346, 'estado': 8526} duplicados: 0
2025 descartes: {'estado': 19792, 'referencia': 7157, 'cvss_network': 9506} duplicados: 0
2026 descartes: {'estado': 40743, 'referencia': 3704, 'cvss_network': 9888} duplicados: 0


Año,Consultados,Elegibles,Seleccionados
2023,30949,11689,250
2024,40704,11130,250
2025,49972,13517,250
2026,76255,21920,250


## 5. Transformación para las siete tablas

Se unifican los CWE por ID y se verifican conflictos. Las debilidades referenciadas ausentes se representan con contexto NULL. Los campos compuestos MITRE conservan su texto original.

Los exploits se deduplican por ID. Cada token `codes` se valida completo tras retirar espacios exteriores; los enlaces fuera de la muestra y los tokens mal formados se reportan. NVD normaliza fechas a UTC y conserva sus cinco estructuras JSON. Las tres tablas puente contienen pares únicos.

In [5]:
preparados = prepare_catalogue(FUENTES, seleccion)
for tabla, registros in preparados.rows.items():
    print(tabla, "filas preparadas:", len(registros))
print("CWE con contexto:", preparados.diagnostics["cwe_with_source_context"])
print("CWE sin contexto:", len(preparados.diagnostics["placeholder_cwe_ids"]))
print("Tokens CVE con anomalías:", len(preparados.diagnostics["malformed_cve_codes"]))
print("CVE de ExploitDB fuera de la muestra:", preparados.diagnostics["exploit_cve_outside_sample_count"])
print("Primer CVE:", preparados.rows["NVD_CVE"][0][:5])

MITRE_CWE filas preparadas: 662
MITRE_CAPEC filas preparadas: 559
NVD_CVE filas preparadas: 1000
EXPLOIT_DB filas preparadas: 46698
CAPEC_CWE filas preparadas: 1212
CVE_CWE filas preparadas: 1170
EXPLOIT_CVE filas preparadas: 7
CWE con contexto: 501
CWE sin contexto: 161
Tokens CVE con anomalías: 25
CVE de ExploitDB fuera de la muestra: 25084
Primer CVE: ('CVE-2018-25085', '2023-05-01T05:15:08.633Z', '2026-06-17T01:54:41.123Z', 'Modified', 'cna@vuldb.com')


## 6. INSERTS parametrizados y carga atómica

Las siguientes sentencias muestran las columnas y el orden real de `executemany`. La celda de carga activa FK, respalda SQLite, comienza `BEGIN IMMEDIATE`, vacía primero asociaciones y después entidades, ejecuta las siete inserciones y valida antes del commit. Cualquier excepción provoca rollback. No se usa `INSERT OR IGNORE`.

In [6]:
for tabla, columnas in INSERT_COLUMNS.items():
    parametros = ", ".join("?" for _ in columnas)
    print(f"INSERT INTO {tabla} ({', '.join(columnas)}) VALUES ({parametros});")

INSERT INTO MITRE_CWE (id, name, description, extended_description, potential_mitigations) VALUES (?, ?, ?, ?, ?);
INSERT INTO MITRE_CAPEC (id, name, description, execution_flow, prerequisites, consequences) VALUES (?, ?, ?, ?, ?, ?);
INSERT INTO NVD_CVE (id, published, last_modified, vuln_status, source_identifier, descriptions_json, metrics_json, weaknesses_json, configurations_json, references_json) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
INSERT INTO EXPLOIT_DB (id, description) VALUES (?, ?);
INSERT INTO CAPEC_CWE (capec_id, cwe_id) VALUES (?, ?);
INSERT INTO CVE_CWE (cve_id, cwe_id) VALUES (?, ?);
INSERT INTO EXPLOIT_CVE (exploit_id, cve_id) VALUES (?, ?);


In [7]:
informe = load_catalogue(BASE, preparados, seleccion, SALIDAS)
print("Carga confirmada. Respaldo:", Path(informe["backup_file"]).name)
print(json.dumps(informe["validation"], ensure_ascii=False, indent=2))

{"phase": "insert", "table": "MITRE_CWE", "rows": 662}


{"phase": "insert", "table": "MITRE_CAPEC", "rows": 559}


{"phase": "insert", "table": "NVD_CVE", "rows": 1000}


{"phase": "insert", "table": "EXPLOIT_DB", "rows": 46698}


{"phase": "insert", "table": "CAPEC_CWE", "rows": 1212}


{"phase": "insert", "table": "CVE_CWE", "rows": 1170}


{"phase": "insert", "table": "EXPLOIT_CVE", "rows": 7}


Carga confirmada. Respaldo: catalogo_20261006T035917975027Z.sqlite3
{
  "table_counts": {
    "MITRE_CWE": 662,
    "MITRE_CAPEC": 559,
    "NVD_CVE": 1000,
    "EXPLOIT_DB": 46698,
    "CAPEC_CWE": 1212,
    "CVE_CWE": 1170,
    "EXPLOIT_CVE": 7
  },
  "catalog_count": 1000,
  "annual_counts": {
    "2023": 250,
    "2024": 250,
    "2025": 250,
    "2026": 250
  },
  "integrity_check": "ok",
  "foreign_key_check": [],
  "coverage": {
    "cve_with_cwe": 971,
    "cve_with_capec": 802,
    "cve_with_exploit": 6,
    "cpe_rows": 9081,
    "cwe_without_context": 161
  }
}


## 7. Consulta del catálogo y aceptación

La vista entrega una fila por CVE con CWE y sus CAPEC anidados y una lista independiente de exploits. No se inventan asociaciones. La vista CPE conserva condiciones ambientales y rutas; la aplicabilidad requiere interpretar también el árbol original.

In [8]:
with closing(sqlite3.connect(BASE)) as conexion:
    conexion.execute("PRAGMA foreign_keys = ON")
    cuotas = conexion.execute("SELECT substr(published,1,4),count(*) FROM NVD_CVE GROUP BY 1 ORDER BY 1").fetchall()
    print("Cuotas almacenadas:", cuotas)
    assert dict(cuotas) == {str(anio): CUOTA_ANUAL for anio in ANIOS}
    assert conexion.execute("PRAGMA integrity_check").fetchone() == ("ok",)
    assert conexion.execute("PRAGMA foreign_key_check").fetchall() == []
    registro = conexion.execute("SELECT id,cwes_json,exploits_json FROM VW_CVE_CATALOGO ORDER BY id LIMIT 1").fetchone()
    debilidades = json.loads(registro[1])
    print("Ejemplo:", registro[0])
    print("CWE y CAPEC:", [{"cwe": d["id"], "contexto": d["context_available"],
                               "capec": [p["id"] for p in d["capec"]]} for d in debilidades])
    print("Exploits:", json.loads(registro[2]))
print("Informes: sqlite/salidas/informe_carga.json y sqlite/salidas/informe_carga.md")

Cuotas almacenadas: [('2023', 250), ('2024', 250), ('2025', 250), ('2026', 250)]
Ejemplo: CVE-2017-20253
CWE y CAPEC: [{'cwe': 'CWE-89', 'contexto': True, 'capec': [7, 66, 108, 109, 110, 470]}]
Exploits: []
Informes: sqlite/salidas/informe_carga.json y sqlite/salidas/informe_carga.md
